# 한국·미국 개별주 후보 스크리너 v3

**운영 방식**: 퀀트로 후보를 뽑고 → AI와 정성 검토(제외 사유 찾기) → 최종 10종목씩 매수.
매번 순수 퀀트 상위 10개를 **그림자 포트폴리오**로 저장해 두고, 정성 검토가 실제로 가치를 더했는지 나중에 비교합니다.

**리밸런싱**: 11/17 · 4/1 · 6/17 · 8/17 (공시 마감 직후). 실행일에 맞는 보고서를 자동 선택합니다.
보유 종목은 후보권(30위) 안이면 유지하고 밖으로 밀린 것만 교체합니다(버퍼 규칙).

| 구분 | 출처 | v2 로직 |
|---|---|---|
| 한국 대형 30 / 소형 30 | 강환국 소형주 가치·성장 | 시장 전체 시총 상위/하위 20% → TTM 재무 → 가치(1/POR·1/PBR·1/PSR) + 퀄리티(영업이익/자산·ROE) + 성장(영업이익증가/자산·매출성장) + 모멘텀(12~7개월) + 반전(최근 1개월, 0.5배) |
| 미국 30 | 켄 피셔 PSR + PRR | 정상마진 이익수익률(평균 영업마진/PSR) + 섹터 내 PSR + R&D/시총(IT·헬스케어) ⊕ 12-1 모멘텀 |
| 미국 ETF 슬리브 | VAA 모멘텀 | QQQM/VEA/VWO 중 1개, 전부 음수면 BIL |

결과물(SAVE_DIR): `KR_대형/소형_..._후보30.csv`, `..._그림자10.csv`, `US_..._후보30.csv`, `AI검토_KR/US_날짜.md` ← 이 md 파일을 AI에게 주고 상의하면 됩니다.

::: {.callout-note}
공개용 정리본입니다. 실행하지 않고 저장된 출력만 보여 줍니다. 투자금 관련 설정값과 그 값이 드러나는 출력은 지웠습니다.
:::


In [1]:
import sys, subprocess
packages = ['opendartreader', 'finance-datareader', 'yfinance', 'tabulate']
res = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + packages,
                     capture_output=True, text=True)
if res.returncode != 0:
    print(res.stdout[-3000:], res.stderr[-3000:])
    raise RuntimeError("패키지 설치 실패. 로그 확인 후 런타임 재시작 후 재시도하세요.")

import os, time, warnings
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import FinanceDataReader as fdr
import yfinance as yf
from tqdm.auto import tqdm
try:
    import OpenDartReader                       # opendartreader <= 0.2.3
except ModuleNotFoundError:
    from opendartreader import OpenDartReader   # opendartreader >= 0.2.4
warnings.filterwarnings('ignore')
pd.set_option('display.width', 250)
print("패키지 로드 완료")

패키지 로드 완료


In [2]:
from google.colab import drive as dr
dr.mount('/content/drive/', force_remount=True)

Mounted at /content/drive/


In [3]:
# ============================================================
# 0. 설정
# ============================================================
try:
    from google.colab import userdata
    DART_API_KEY = userdata.get('DART_API_KEY')
except Exception:
    import getpass
    DART_API_KEY = getpass.getpass('DART API KEY 입력: ')

SAVE_DIR = '/content/drive/MyDrive/Investment/Rebalancing_portfolio'
os.makedirs(SAVE_DIR, exist_ok=True)

RUN_DATE = datetime.now()
SNAP_STR = RUN_DATE.strftime('%Y%m%d')

# ── 한국 ─────────────────────────────────────────────
KR_SIZE_Q        = 0.20             # 시장 전체 시총 상위/하위 20%
KR_N_CANDIDATES  = 30               # 대형/소형 각각 후보 수 (AI 정성검토용)
KR_N_SHADOW      = 10               # 그림자(순수 퀀트) 포트폴리오 종목 수
KR_MAX_PER_SECTOR = 4               # 후보 30개 안에서 동일 업종 최대 개수
KR_MIN_PRICE     = 1_000            # 동전주 배제
KR_MIN_MARCAP    = 10_000_000_000   # 소형주 최소 시총 100억
KR_POSITION_SIZE = None  # 비공개: 실제 값은 원본 노트북(private)에만 둠
KR_MAX_PCT_OF_TV = 0.02             # 종목당 투자금이 20일 평균 거래대금의 2%를 넘지 않게
KR_MIN_TV20      = max(KR_POSITION_SIZE / KR_MAX_PCT_OF_TV, 30_000_000)
# (원본의 '3억'은 시장 하위 20%(시총 ~370억 이하) 소형주 대부분을 탈락시킴: 흑자 소형주 중앙값 ~0.4억)
KR_ONE_OFF_RATIO = 2.0              # 순이익(TTM) > 영업이익(TTM) x 2 → 일회성 이익 의심, 제외
DART_SLEEP       = 0.5

# 팩터 정의 (모두 높을수록 좋음, reversal만 낮을수록 좋음)
KR_FACTORS = {
    'value':    [('OP_P', True), ('B_P', True), ('S_P', True)],   # 1/POR, 1/PBR, 1/PSR
    'quality':  [('OP_A', True), ('ROE', True)],                  # 영업이익/자산, ROE
    'growth':   [('dOP_A', True), ('rev_growth', True)],          # 영업이익 증가액/자산, 매출 성장률
    'momentum': [('ret_12_7', True)],                             # 12~7개월 전 수익률
    'reversal': [('ret_1m', False)],                              # 최근 1개월 급등 감점
}
KR_WEIGHTS = {'value': 1.0, 'quality': 1.0, 'growth': 1.0, 'momentum': 1.0, 'reversal': 0.5}

# ── 미국 ─────────────────────────────────────────────
US_EXCLUDE_SECTORS = ['Financials', 'Utilities', 'Real Estate']   # PSR이 의미 없는 업종
US_RD_SECTORS      = ['Information Technology', 'Health Care']    # PRR 적용 업종
US_MIN_AVG_MARGIN  = 0.05           # 평균 영업마진 5% 미만 제외 (피셔: 장기 마진 확인)
US_MIN_YEARS       = 3              # 평균 마진 계산 최소 연도 수
US_N_CANDIDATES    = 30
US_N_SHADOW        = 10
US_MAX_PER_SECTOR  = 6              # 후보 30개 안에서 동일 섹터 최대 개수
US_FACTORS = {
    'value':    [('NEY', True), ('SP_sector', True), ('RDY', True)],
    'momentum': [('ret_12_1', True)],
}
US_WEIGHTS = {'value': 1.0, 'momentum': 1.0}


def select_report(today):
    """실행일 기준 '제출 마감이 지난 가장 최신 보고서'"""
    y, md = today.year, (today.month, today.day)
    if md < (4, 1):   return y - 1, '11014', '3분기보고서'
    if md < (5, 16):  return y - 1, '11011', '사업보고서'
    if md < (8, 15):  return y,     '11013', '1분기보고서'
    if md < (11, 15): return y,     '11012', '반기보고서'
    return y, '11014', '3분기보고서'

YEAR, REPRT, REPORT_NAME = select_report(RUN_DATE)
IS_ANNUAL = (REPRT == '11011')
print(f"실행일 {RUN_DATE:%Y-%m-%d} → {YEAR}년 {REPORT_NAME}(reprt_code={REPRT})"
      + ("" if IS_ANNUAL else f" + {YEAR-1}년 사업보고서로 TTM 계산"))

실행일 2026-11-02 → 2026년 반기보고서(reprt_code=11012) + 2025년 사업보고서로 TTM 계산


In [4]:
# ============================================================
# 공통 계산 함수 (데이터 소스와 무관한 순수 함수 - 단위 테스트 가능)
# ============================================================
import re
import numpy as np
import pandas as pd


def safe_num(val):
    """문자열 → 숫자 (실패/빈칸/'-' → NaN)"""
    try:
        s = str(val).replace(',', '').strip()
        if s in ('', '-', 'nan', 'None', 'NaN'):
            return np.nan
        return float(s)
    except Exception:
        return np.nan


# ── DART 단일회사 주요계정(finstate) 파싱 ─────────────────────
ACCOUNT_CANDIDATES = {
    'rev':    ['매출액', '수익(매출액)', '영업수익', '매출'],
    'op':     ['영업이익', '영업이익(손실)'],
    'ni':     ['당기순이익', '당기순이익(손실)', '반기순이익', '반기순이익(손실)',
               '분기순이익', '분기순이익(손실)', '연결당기순이익', '순이익'],
    'assets': ['자산총계'],
    'equity': ['자본총계'],
}
# 부분일치 시 잘못 걸리는 계정 배제 (예: '순이익' → '법인세차감전 순이익')
ACCOUNT_EXCLUDE = {
    'rev': ['원가', '총이익'],
    'ni':  ['법인세', '차감전', '비지배'],
}
IS_KEYS = ('rev', 'op', 'ni')


def _pick_row(sub, key):
    names = sub['account_nm'].astype(str).str.strip()
    excl = ACCOUNT_EXCLUDE.get(key, [])
    ok = ~names.str.contains('|'.join(map(re.escape, excl)), na=False) if excl else names.notna()
    cands = ACCOUNT_CANDIDATES[key]
    for c in cands:                                   # 1순위: 정확 일치
        hit = sub[(names == c) & ok]
        if len(hit):
            return hit.iloc[0]
    for c in cands:                                   # 2순위: 부분 일치
        hit = sub[names.str.contains(re.escape(c), na=False) & ok]
        if len(hit):
            return hit.iloc[0]
    return None


def _val(row, field):
    return safe_num(row[field]) if field in row.index else np.nan


def parse_finstate(df_, reprt_code, fs_order=('CFS', 'OFS')):
    """finstate 결과 → 평탄화 dict.
    손익(IS): *_cum = 당기 누적, *_cum_prev = 전년 동기 누적 (사업보고서면 연간/전년 연간)
    재무상태(BS): assets, equity = 당기말 잔액
    cum_fallback=True: 반기/3분기인데 누적(add_amount)이 비어 3개월치로 대체된 경우 (점검 필요)
    """
    if df_ is None or len(df_) == 0 or 'fs_div' not in df_.columns:
        return {}
    quarterly_cum = reprt_code in ('11012', '11014')
    for fs in fs_order:
        sub = df_[df_['fs_div'] == fs]
        if len(sub) == 0:
            continue
        out, fallback = {'fs_div': fs}, False
        for key in ACCOUNT_CANDIDATES:
            row = _pick_row(sub, key)
            if row is None:
                continue
            if key in IS_KEYS:
                cur = _val(row, 'thstrm_add_amount')
                if np.isnan(cur):
                    cur = _val(row, 'thstrm_amount')
                    fallback = fallback or quarterly_cum
                prev = _val(row, 'frmtrm_add_amount')
                if np.isnan(prev):
                    prev = _val(row, 'frmtrm_amount')
                out[f'{key}_cum'], out[f'{key}_cum_prev'] = cur, prev
            else:
                out[key] = _val(row, 'thstrm_amount')
        out['cum_fallback'] = fallback
        if not np.isnan(out.get('rev_cum', np.nan)):
            return out
    return {}


def add_ttm(d, is_annual_report):
    """TTM(최근 12개월) = 직전 사업연도 + 당기 누적 - 전년 동기 누적.
    사업보고서 시점이면 당기 누적 자체가 연간 = TTM.
    d 컬럼: {rev,op,ni}_cum, {rev,op,ni}_cum_prev, ann_{rev,op,ni}"""
    d = d.copy()
    for k in IS_KEYS:
        if is_annual_report:
            d[f'{k}_ttm'] = d[f'{k}_cum']
        else:
            d[f'{k}_ttm'] = d[f'ann_{k}'] + d[f'{k}_cum'] - d[f'{k}_cum_prev']
    return d


def add_kr_metrics(d):
    """가치/퀄리티/이익개선 지표. 모두 '높을수록 좋음' 방향으로 정의"""
    d = d.copy()
    mc = d['market_cap']
    d['OP_P'] = d['op_ttm'] / mc            # 1/POR (영업이익 수익률)
    d['B_P']  = d['equity'] / mc            # 1/PBR
    d['S_P']  = d['rev_ttm'] / mc           # 1/PSR
    d['E_P']  = d['ni_ttm'] / mc            # 1/PER (표시/참고용)
    d['OP_A'] = d['op_ttm'] / d['assets']   # 영업이익/자산 (operating profitability)
    d['ROE']  = d['ni_ttm'] / d['equity']
    d['op_margin'] = d['op_ttm'] / d['rev_ttm']
    # 이익개선: 같은 기간 누적끼리 비교 → 계절성 없음. 자산으로 나눠 기저효과(작은 분모) 제거
    d['dOP_A'] = (d['op_cum'] - d['op_cum_prev']) / d['assets']
    d['rev_growth'] = np.where(d['rev_cum_prev'] > 0,
                               d['rev_cum'] / d['rev_cum_prev'] - 1, np.nan)
    # 표시용 전통 배수
    d['PER'] = mc / d['ni_ttm']
    d['PBR'] = mc / d['equity']
    d['PSR'] = mc / d['rev_ttm']
    d['POR'] = mc / d['op_ttm']
    return d


# ── 가격 기반 지표 ───────────────────────────────────────────
D1M, D6M, D12M = 21, 126, 252


def price_features(px):
    """px: DataFrame(Close, Volume), 날짜 오름차순.
    ret_12_7 : 12~7개월 전 구간 수익률 (한국에서 유효한 중기 모멘텀)
    ret_12_1 : 표준 모멘텀 (최근 1개월 제외) - 미국용/참고용
    ret_1m   : 최근 1개월 (한국은 단기 반전 → 낮을수록 유리)"""
    out = {'n_days': 0, 'tv20': np.nan, 'ret_1m': np.nan,
           'ret_12_7': np.nan, 'ret_12_1': np.nan}
    if px is None or len(px) == 0:
        return out
    c = px['Close'].astype(float)
    c = c[c > 0].dropna()
    out['n_days'] = len(c)
    if 'Volume' in px.columns:
        tv = (px['Close'] * px['Volume']).dropna()
        out['tv20'] = float(tv.tail(20).mean()) if len(tv) else np.nan
    if len(c) > D1M:
        out['ret_1m'] = c.iloc[-1] / c.iloc[-1 - D1M] - 1
    if len(c) > D12M:
        out['ret_12_7'] = c.iloc[-1 - D6M] / c.iloc[-1 - D12M] - 1
        out['ret_12_1'] = c.iloc[-1 - D1M] / c.iloc[-1 - D12M] - 1
    return out


# ── 통합 점수 ───────────────────────────────────────────────
def composite_scores(d, factors, weights):
    """factors: {그룹: [(컬럼, 높을수록좋음?), ...]}
    각 지표 → 유니버스 내 백분위(0~1) → 그룹 평균(S_그룹) → 가중평균(S_total).
    지표가 결측이면 그 그룹은 나머지 지표 평균, 그룹 전체가 결측이면 중립 0.5."""
    d = d.copy()
    for grp, items in factors.items():
        cols = []
        for col, higher_better in items:
            s = d[col] if higher_better else -d[col]
            d[f's_{col}'] = s.rank(pct=True, method='average')
            cols.append(f's_{col}')
        d[f'S_{grp}'] = d[cols].mean(axis=1, skipna=True).fillna(0.5)
    w = {g: float(v) for g, v in weights.items() if v and g in factors}
    tot = sum(w.values())
    d['S_total'] = sum(d[f'S_{g}'] * v for g, v in w.items()) / tot
    d['rank_total'] = d['S_total'].rank(ascending=False, method='min').astype(int)
    return d.sort_values('S_total', ascending=False)


def pick_with_cap(d, n, group_col=None, max_per=None, score_col='S_total'):
    """점수 순으로 n개 선택하되 같은 그룹(업종/섹터)은 max_per개까지"""
    d = d.sort_values(score_col, ascending=False)
    if not group_col or not max_per or group_col not in d.columns:
        return d.head(n)
    counts, keep = {}, []
    for idx, g in d[group_col].items():
        key = g if (isinstance(g, str) and g.strip()) else f'__na_{idx}'
        if counts.get(key, 0) < max_per:
            keep.append(idx)
            counts[key] = counts.get(key, 0) + 1
        if len(keep) >= n:
            break
    return d.loc[keep]


# ── 미국: 연간 손익(yfinance financials) 정리 ─────────────────
def us_fin_summary(fin, min_years=3):
    """fin: yfinance Ticker.financials (index=계정, columns=회계연도말, 최신이 앞)
    avg_op_margin: 가용 연도(최대 4~5년) 평균 영업마진 → 피셔의 '정상 마진'
    rd_last: 최신 회계연도 R&D (없으면 0)"""
    out = {'avg_op_margin': np.nan, 'n_years': 0, 'rd_last': 0.0, 'last_fy': None}
    if fin is None or fin.empty:
        return out

    def row(keys):
        for k in keys:
            if k in fin.index:
                return pd.to_numeric(fin.loc[k], errors='coerce')
        return None

    rev = row(['Total Revenue', 'Operating Revenue'])
    op = row(['Operating Income', 'Total Operating Income As Reported'])
    rd = row(['Research And Development', 'Research Development', 'ResearchAndDevelopment'])
    try:
        out['last_fy'] = pd.to_datetime(fin.columns[0]).date()
    except Exception:
        pass
    if rev is not None and op is not None:
        m = (op / rev.where(rev > 0)).dropna()
        out['n_years'] = int(len(m))
        if len(m) >= min_years:
            out['avg_op_margin'] = float(m.mean())
    if rd is not None:
        rdv = rd.dropna()
        if len(rdv):
            out['rd_last'] = float(abs(rdv.iloc[0]))
    return out

## 한국

In [5]:
# ============================================================
# [KR 1] 유니버스: 시장 전체 기준 시총 상위/하위 20%
# ============================================================
# 원본과의 차이: 재무 필터를 통과한 종목 중 하위 20%가 아니라, 시장 전체(보통주, 금융/지주/스팩/리츠/
# 관리종목 제외)에서 먼저 시총 구간을 자른다 → 분기마다 '소형주' 정의가 흔들리지 않음.
# 덤으로 DART 조회 대상이 전체의 40%로 줄어 수집 시간이 단축된다.
print("[KR 1] 유니버스 구성...")

import glob
MIN_LISTED = 1500   # 정상 시장목록이면 시총>0 종목이 2,000개 이상. 이보다 적으면 비정상으로 판단

def _snap_ok(m):
    return m is not None and 'market_cap' in m.columns and int((m['market_cap'] > 0).sum()) >= MIN_LISTED

def _load_snap(path):
    try:
        return pd.read_csv(path, dtype={'ticker': str})
    except Exception:
        return None

mkt_cache = f'{SAVE_DIR}/mkt_snapshot_{SNAP_STR}.csv'
mkt_df = _load_snap(mkt_cache) if os.path.exists(mkt_cache) else None
if _snap_ok(mkt_df):
    print(f"  캐시 로드: {mkt_cache}")
else:
    if mkt_df is not None:
        print(f"  ⚠ 캐시가 비정상(시총>0 종목 {int((mkt_df['market_cap'] > 0).sum()) if 'market_cap' in mkt_df else 0}개) → 다시 받습니다")
    frames = []
    for mk in ['KOSPI', 'KOSDAQ']:
        info = fdr.StockListing(mk).copy()
        info['market'] = mk
        frames.append(info)
    lst = pd.concat(frames, ignore_index=True)
    lst = lst.rename(columns={'Code': 'ticker', 'Name': 'name', 'Close': 'close',
                              'Marcap': 'market_cap', 'Stocks': 'shares',
                              'Volume': 'volume', 'Dept': 'dept'})
    if 'dept' not in lst.columns:
        lst['dept'] = ''
    lst['ticker'] = lst['ticker'].astype(str).str.zfill(6)
    mkt_df = lst[['ticker', 'name', 'market', 'dept', 'close', 'market_cap', 'shares', 'volume']].copy()
    mkt_df = mkt_df[mkt_df['market_cap'] > 0].reset_index(drop=True)

    # 업종/주요제품 (KIND). 실패해도 진행 - 업종캡만 비활성화됨
    try:
        desc = fdr.StockListing('KRX-DESC')[['Code', 'Sector', 'Industry']]
        desc = desc.rename(columns={'Code': 'ticker', 'Sector': 'sector', 'Industry': 'products'})
        desc['ticker'] = desc['ticker'].astype(str).str.zfill(6)
        mkt_df = mkt_df.merge(desc, on='ticker', how='left')
    except Exception as e:
        print(f"  ⚠ 업종정보(KRX-DESC) 조회 실패 → 업종캡 없이 진행: {type(e).__name__}")
        mkt_df['sector'], mkt_df['products'] = np.nan, np.nan

    if _snap_ok(mkt_df):
        mkt_df.to_csv(mkt_cache, index=False, encoding='utf-8-sig')
        print(f"  저장: {mkt_cache}")
    else:
        # FDR/KRX 시세 응답이 간헐적으로 비거나 0으로 옴(장 시작 전·장중 모두 관찰) → 직전 정상 스냅샷 사용
        print(f"  ⚠ 시장목록 비정상(시총>0 종목 {len(mkt_df)}개) - KRX 시세 응답 오류")
        prev = []
        for f in sorted(glob.glob(f'{SAVE_DIR}/mkt_snapshot_*.csv'), reverse=True):
            if f == mkt_cache:
                continue
            m = _load_snap(f)
            if _snap_ok(m):
                prev = [f, m]
                break
        if not prev:
            raise RuntimeError("정상 시장목록을 받지 못했고 이전 스냅샷도 없습니다. "
                               "잠시 후 다시 실행하세요.")
        print(f"  → 직전 정상 스냅샷 사용: {os.path.basename(prev[0])}")
        mkt_df = prev[1]

mkt_df = mkt_df[mkt_df['market_cap'] > 0].reset_index(drop=True)


def fix_sector_cols(m):
    """KIND 목록 형식이 바뀌어 'Sector'에 소속부(중견기업부 등), 'Industry'에 업종이 들어오는 경우 교정"""
    s = m['sector'].dropna().astype(str)
    p = m['products'].dropna().astype(str)
    s_is_dept = len(s) and s.str.endswith('부').mean() > 0.5
    p_is_ksic = len(p) and p.str.endswith('업').mean() > 0.5
    if s_is_dept and p_is_ksic:
        m['sector'], m['products'] = m['products'], np.nan
        print("  (업종 열 교정: KIND 형식 변경 → 'Industry'를 업종으로 사용, 주요제품 정보 없음)")
    return m

mkt_df = fix_sector_cols(mkt_df)

# DART 고유번호 매핑 (보통주만 남음: 우선주/일부 리츠 등은 매핑 실패로 자연 제외)
dart = OpenDartReader(DART_API_KEY)
corp = dart.corp_codes.copy()
corp = corp[corp['stock_code'].notna()].copy()
corp['stock_code'] = corp['stock_code'].astype(str).str.strip().str.zfill(6)
uni = mkt_df.merge(corp[['corp_code', 'stock_code']], left_on='ticker',
                   right_on='stock_code', how='inner').drop(columns='stock_code')
print(f"  DART 고유번호: 전체 {len(dart.corp_codes)}개 중 상장 {len(corp)}개 → 시장목록과 매핑 {len(uni)}개")
if len(uni) == 0:
    print(dart.corp_codes.head())
    raise RuntimeError("DART 고유번호 매핑 0개 - 위 corp_codes 샘플(stock_code 형식)을 확인하세요.")

def log_step(d, mask, label):
    out = d[mask]
    print(f"  {label:<34s}: {len(d):5d} → {len(out):5d}")
    return out

print(f"  {'KOSPI+KOSDAQ, DART 매핑 보통주':<34s}: {len(uni):5d}")
NAME_EXCL   = ['스팩', '리츠', 'REIT', '인프라', '맥쿼리', '지주', '홀딩스', '홀딩',
               '은행', '보험', '증권', '캐피탈', '카드', '저축', '금융']
SECTOR_EXCL = ['금융', '은행', '보험', '증권', '신탁', '저축', '투자업']
uni = log_step(uni, ~uni['name'].str.contains('|'.join(NAME_EXCL), na=False), '이름 기준 금융/지주/스팩/리츠 제외')
uni = log_step(uni, ~uni['sector'].fillna('').str.contains('|'.join(SECTOR_EXCL)), '업종 기준 금융업 제외')
uni = log_step(uni, ~uni['dept'].fillna('').str.contains('관리|환기|정리'), '관리/환기/정리매매 종목 제외')

q_hi, q_lo = uni['market_cap'].quantile(1 - KR_SIZE_Q), uni['market_cap'].quantile(KR_SIZE_Q)
uni['size_group'] = np.where(uni['market_cap'] >= q_hi, '대형',
                     np.where(uni['market_cap'] <= q_lo, '소형', None))
pool = uni[uni['size_group'].notna()].copy()
pool = pool[~((pool['size_group'] == '소형') &
              ((pool['market_cap'] < KR_MIN_MARCAP) | (pool['close'] < KR_MIN_PRICE)))]
print(f"\n  대형 기준: 시총 {q_hi/1e8:,.0f}억 이상 → {int((pool.size_group=='대형').sum())}개")
print(f"  소형 기준: 시총 {q_lo/1e8:,.0f}억 이하 (100억 이상, 1,000원 이상) → {int((pool.size_group=='소형').sum())}개")
if len(pool) == 0:
    raise RuntimeError("대형/소형 풀이 비었습니다 - 위 단계별 로그를 확인하세요.")

[KR 1] 유니버스 구성...
  ⚠ 캐시가 비정상(시총>0 종목 0개) → 다시 받습니다
  ⚠ 시장목록 비정상(시총>0 종목 0개) - KRX 시세 응답 오류
  → 직전 정상 스냅샷 사용: mkt_snapshot_20261101.csv
  (업종 열 교정: KIND 형식 변경 → 'Industry'를 업종으로 사용, 주요제품 정보 없음)
  DART 고유번호: 전체 119879개 중 상장 119879개 → 시장목록과 매핑 2655개
  KOSPI+KOSDAQ, DART 매핑 보통주         :  2655
  이름 기준 금융/지주/스팩/리츠 제외              :  2655 →  2439
  업종 기준 금융업 제외                      :  2439 →  2360
  관리/환기/정리매매 종목 제외                  :  2360 →  2202

  대형 기준: 시총 3,766억 이상 → 441개
  소형 기준: 시총 374억 이하 (100억 이상, 1,000원 이상) → 402개


In [6]:
# ============================================================
# [KR 2] DART 재무 수집 → TTM
# ============================================================
# - 당기 보고서(누적) + 직전 사업보고서(연간)로 TTM 계산 → 반기 x2 같은 연환산의 계절성 왜곡 제거
# - 종목 단위 증분 캐시: 같은 보고서 기준이면 이미 받은 종목은 건너뜀 (중간에 끊겨도 이어서 수집)
cache_file = f'{SAVE_DIR}/dart_ttm_{YEAR}_{REPRT}.csv'
cache = (pd.read_csv(cache_file, dtype={'ticker': str, 'corp_code': str})
         if os.path.exists(cache_file) else pd.DataFrame(columns=['ticker', 'ok']))
done = set(cache.loc[cache['ok'].astype(str) == 'True', 'ticker'])   # 실패 종목은 매번 재시도
todo = pool[~pool['ticker'].isin(done)]
calls = len(todo) * (1 if IS_ANNUAL else 2)
print(f"[KR 2] DART 수집: 대상 {len(pool)}개 중 캐시 {len(pool)-len(todo)}개, "
      f"신규 {len(todo)}개 (API 약 {calls}회, ~{calls*(DART_SLEEP+0.3)/60:.0f}분)")


import contextlib, io
def _quiet(fn, *a, **k):
    """opendartreader가 데이터 없을 때 찍는 {'status': '013'...} 메시지 숨김"""
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*a, **k)

def fetch_one(corp_code):
    try:
        cur = parse_finstate(_quiet(dart.finstate, corp_code, YEAR, reprt_code=REPRT), REPRT)
    except Exception:
        return {}
    if not cur or IS_ANNUAL:
        return cur
    time.sleep(DART_SLEEP)
    try:   # 직전 사업보고서: 당기 보고서와 같은 연결/별도 기준으로
        ann = parse_finstate(_quiet(dart.finstate, corp_code, YEAR - 1, reprt_code='11011'),
                             '11011', fs_order=(cur['fs_div'],))
    except Exception:
        ann = {}
    for k in IS_KEYS:
        cur[f'ann_{k}'] = ann.get(f'{k}_cum', np.nan)
    return cur


# ── 사전 점검: 첫 종목 1개로 DART 응답 형식 확인 (API 키/한도/형식 문제를 30분 돌리기 전에 발견)
if len(todo):
    _c = todo.iloc[0]
    _raw = dart.finstate(_c['corp_code'], YEAR, reprt_code=REPRT)
    print(f"  사전점검 {_c['name']}: 응답 {0 if _raw is None else len(_raw)}행"
          + (f", 열 {list(_raw.columns)[:12]}" if _raw is not None and len(_raw) else ""))
    _p = fetch_one(_c['corp_code'])
    print(f"  파싱 결과: {_p}")
    if not _p or np.isnan(_p.get('ann_rev', 0.0) if not IS_ANNUAL else 0.0):
        print("  ⚠ 첫 종목 파싱 실패/연간값 없음 - 아래 수집 결과의 실패 수를 확인하세요 "
              "(위에 {'status': ...} 메시지가 있으면 그 내용이 원인입니다: 010=키 오류, 020=일일 한도 초과)")

new_rows, fails = [], []
for i, (_, r) in enumerate(todo.iterrows(), 1):
    rec = fetch_one(r['corp_code'])
    if not rec:                                 # 일시 오류 대비 1회 재시도
        time.sleep(1.0)
        rec = fetch_one(r['corp_code'])
    rec['ticker'] = r['ticker']
    rec['ok'] = bool(rec.get('fs_div'))
    new_rows.append(rec)
    if not rec['ok']:
        fails.append(r['ticker'])
    if i % 100 == 0 or i == len(todo):
        print(f"  {i:4d}/{len(todo)}  실패 누적 {len(fails)}")
        pd.concat([cache, pd.DataFrame(new_rows)], ignore_index=True).to_csv(cache_file, index=False)
    time.sleep(DART_SLEEP)

fs_df = pd.concat([cache, pd.DataFrame(new_rows)], ignore_index=True).drop_duplicates('ticker', keep='last')
print(f"  재무 확보 {int(fs_df['ok'].astype(bool).sum())}개 / 실패 {int((~fs_df['ok'].astype(bool)).sum())}개 → {cache_file}")

[KR 2] DART 수집: 대상 843개 중 캐시 825개, 신규 18개 (API 약 36회, ~0분)
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

  사전점검 금양: 응답 0행
  파싱 결과: {}
  ⚠ 첫 종목 파싱 실패/연간값 없음 - 아래 수집 결과의 실패 수를 확인하세요 (위에 {'status': ...} 메시지가 있으면 그 내용이 원인입니다: 010=키 오류, 020=일일 한도 초과)
    18/18  실패 누적 18
  재무 확보 859개 / 실패 30개 → /content/drive/MyDrive/Investment/Rebalancing_portfolio/dart_ttm_2026_11012.csv


In [7]:
# ============================================================
# [KR 3] 지표 계산 + 기본 필터
# ============================================================
print("[KR 3] 지표 계산 및 필터...")
num_cols = [f'{k}_{s}' for k in IS_KEYS for s in ('cum', 'cum_prev')] + \
           [f'ann_{k}' for k in IS_KEYS] + ['assets', 'equity']
fs_use = fs_df[fs_df['ok'].astype(str) == 'True'].copy()
for c in num_cols:
    if c not in fs_use.columns:
        fs_use[c] = np.nan
    fs_use[c] = pd.to_numeric(fs_use[c], errors='coerce')

kr = pool.merge(fs_use[['ticker', 'fs_div', 'cum_fallback'] + num_cols], on='ticker', how='inner')
kr = add_kr_metrics(add_ttm(kr, IS_ANNUAL))

print(f"  {'재무 확보':<34s}: {len(kr):5d}")
if len(kr):
    print("  결측 수: " + ", ".join(f"{c} {int(kr[c].isna().sum())}" for c in
                                  ['rev_cum', 'op_cum', 'ni_cum', 'ann_rev', 'ann_op', 'ann_ni', 'assets', 'equity']
                                  if c in kr.columns))
_n0 = len(kr)
kr = log_step(kr, kr[['rev_ttm', 'op_ttm', 'ni_ttm', 'equity', 'assets']].notna().all(axis=1), 'TTM 계산 가능')
kr = log_step(kr, (kr['rev_ttm'] > 0) & (kr['equity'] > 0), '매출 > 0, 자본 > 0')
kr = log_step(kr, (kr['op_ttm'] > 0) & (kr['ni_ttm'] > 0), '영업이익 > 0, 순이익 > 0 (TTM)')
kr = log_step(kr, kr['ni_ttm'] <= KR_ONE_OFF_RATIO * kr['op_ttm'],
              f'일회성 이익 의심 제외 (순이익 ≤ 영업이익x{KR_ONE_OFF_RATIO:g})')
if len(kr) == 0:
    raise RuntimeError(f"KR 3 결과 0개 (재무 확보 {_n0}개) - 위 단계별 로그에서 어느 필터에서 0이 됐는지 확인하세요.")
n_fb = int(kr['cum_fallback'].astype(str).eq('True').sum())
if n_fb:
    print(f"  ⚠ 누적금액 결측으로 3개월치를 쓴 종목 {n_fb}개 → 결과표 '점검' 열에 표시")

[KR 3] 지표 계산 및 필터...
  재무 확보                             :   825
  결측 수: rev_cum 0, op_cum 0, ni_cum 4, ann_rev 13, ann_op 13, ann_ni 17, assets 0, equity 2
  TTM 계산 가능                         :   825 →   805
  매출 > 0, 자본 > 0                    :   805 →   805
  영업이익 > 0, 순이익 > 0 (TTM)           :   805 →   471
  일회성 이익 의심 제외 (순이익 ≤ 영업이익x2)       :   471 →   422


In [8]:
# ============================================================
# [KR 4] 가격 지표: 20일 거래대금, 모멘텀(12~7개월), 최근 1개월
# ============================================================
px_cache = f'{SAVE_DIR}/kr_price_feat_{SNAP_STR}.csv'
pf = (pd.read_csv(px_cache, dtype={'ticker': str}) if os.path.exists(px_cache)
      else pd.DataFrame(columns=['ticker']))
todo = [t for t in kr['ticker'] if t not in set(pf['ticker'])]
print(f"[KR 4] 가격 조회 {len(todo)}개 (캐시 {len(kr)-len(todo)}개)...")
start_str = (RUN_DATE - timedelta(days=400)).strftime('%Y-%m-%d')
rows = []
for i, t in enumerate(todo, 1):
    try:
        px = fdr.DataReader(t, start_str)
    except Exception:
        px = None
    rows.append({'ticker': t, **price_features(px)})
    if i % 100 == 0:
        print(f"  {i}/{len(todo)}")
    time.sleep(0.05)
if rows:
    pf = pd.concat([pf, pd.DataFrame(rows)], ignore_index=True)
    pf.to_csv(px_cache, index=False)

kr = kr.drop(columns=[c for c in pf.columns if c != 'ticker' and c in kr.columns]) \
       .merge(pf, on='ticker', how='left')
for c in ['n_days', 'tv20', 'ret_1m', 'ret_12_7', 'ret_12_1']:
    if c not in kr.columns:
        kr[c] = np.nan
kr = log_step(kr, kr['tv20'].fillna(0) >= KR_MIN_TV20,
              f'20일 평균 거래대금 {KR_MIN_TV20/1e8:.1f}억 이상 (종목당 {KR_POSITION_SIZE/1e4:,.0f}만원 기준)')
for _g in ['대형', '소형']:
    _n = int((kr['size_group'] == _g).sum())
    print(f"    {_g}: {_n}개" + (f"  ⚠ 후보 {KR_N_CANDIDATES}개보다 적음" if _n < KR_N_CANDIDATES else ""))
n_nomom = int(kr['ret_12_7'].isna().sum())
if n_nomom:
    print(f"  (상장 1년 미만 등 모멘텀 계산 불가 {n_nomom}개 → 모멘텀 점수 중립 0.5 처리)")

[KR 4] 가격 조회 0개 (캐시 422개)...
  20일 평균 거래대금 하한 이상 (종목당 투자금 기준, 금액 비공개):   422 →   353
    대형: 312개
    소형: 41개
  (상장 1년 미만 등 모멘텀 계산 불가 4개 → 모멘텀 점수 중립 0.5 처리)


In [9]:
# ============================================================
# [KR 5] 점수화 → 대형/소형 후보 30개 + 그림자 포트폴리오 + 희석 공시 점검
# ============================================================
# 점수: 각 지표를 '같은 규모 그룹 안에서' 백분위로 바꾼 뒤 그룹 평균 → 가중 평균 (KR_WEIGHTS)
#   value    : 1/POR, 1/PBR, 1/PSR                       (강환국 가치 계열, PER 대신 POR로 일회성 영향 축소)
#   quality  : 영업이익/자산, ROE
#   growth   : 영업이익 증가액/자산, 매출 성장률         (강환국 성장 계열, 성장률% 대신 증가액/자산)
#   momentum : 12~7개월 전 구간 수익률                   (한국은 표준 12-1 모멘텀이 약함)
#   reversal : 최근 1개월 수익률 (낮을수록 유리, 가중치 0.5)
KR_SHADOW_MAX_PER_SECTOR = 2

def dilution_check(corp_code, days=365):
    """최근 1년 주요사항보고서 중 CB/BW/EB 발행, 유상증자, 감자 건수"""
    try:
        start = (RUN_DATE - timedelta(days=days)).strftime('%Y-%m-%d')
        import contextlib, io
        with contextlib.redirect_stdout(io.StringIO()):     # '조회된 데이타가 없습니다' 출력 숨김
            lst = dart.list(corp_code, start=start, end=RUN_DATE.strftime('%Y-%m-%d'), kind='B')
        if lst is None or len(lst) == 0:
            return ''
        names = lst['report_nm'].astype(str)
        tags = []
        for kw, tag in [('전환사채', 'CB'), ('신주인수권부사채', 'BW'), ('교환사채', 'EB'),
                        ('유상증자', '유증'), ('감자', '감자')]:
            n = int(names.str.contains(kw).sum())
            if n:
                tags.append(f'{tag}{n}')
        return ' '.join(tags)
    except Exception:
        return '조회실패'

KR_SHOW = {
    'rank_total': '순위', 'ticker': '종목코드', 'name': '종목명', 'market': '시장', 'sector': '업종',
    'products': '주요제품', 'mcap_억': '시총(억)', 'PER': 'PER', 'PBR': 'PBR', 'PSR': 'PSR', 'POR': 'POR',
    'ROE_%': 'ROE(%)', 'opm_%': '영업이익률(%)', 'revg_%': '매출성장(%)', 'dOP_A_%': '영업이익증가/자산(%)',
    'm12_7_%': '12-7개월(%)', 'm1_%': '최근1개월(%)', 'tv20_억': '거래대금20d(억)',
    'S_value': '가치', 'S_quality': '퀄리티', 'S_growth': '성장', 'S_momentum': '모멘텀',
    'S_reversal': '반전', 'S_total': '총점', 'dilution': '최근1년 희석공시', 'check': '점검',
}

def kr_table(d):
    t = d.copy()
    t['mcap_억'] = (t['market_cap'] / 1e8).round(0)
    for a, b in [('ROE', 'ROE_%'), ('op_margin', 'opm_%'), ('rev_growth', 'revg_%'),
                 ('dOP_A', 'dOP_A_%'), ('ret_12_7', 'm12_7_%'), ('ret_1m', 'm1_%')]:
        t[b] = (t[a] * 100).round(1)
    t['tv20_억'] = (t['tv20'] / 1e8).round(1)
    for c in ['PER', 'PBR', 'PSR', 'POR']:
        t[c] = t[c].round(2)
    for c in [c for c in t.columns if c.startswith('S_')]:
        t[c] = t[c].round(2)
    chk = []
    for _, r in t.iterrows():
        f = []
        if str(r.get('cum_fallback')) == 'True': f.append('누적값결측')
        if pd.isna(r.get('ret_12_7')):           f.append('상장1년미만')
        chk.append(','.join(f))
    t['check'] = chk
    cols = [c for c in KR_SHOW if c in t.columns]
    return t[cols].rename(columns=KR_SHOW).reset_index(drop=True)

kr_results = {}
for grp in ['대형', '소형']:
    g = kr[kr['size_group'] == grp]
    if len(g) == 0:
        print(f"  {grp}: 종목 없음"); continue
    sc = composite_scores(g, KR_FACTORS, KR_WEIGHTS)
    cand = pick_with_cap(sc, KR_N_CANDIDATES, 'sector', KR_MAX_PER_SECTOR).copy()
    shadow = pick_with_cap(sc, KR_N_SHADOW, 'sector', KR_SHADOW_MAX_PER_SECTOR).copy()
    print(f"\n  {grp}: 점수화 {len(sc)}개 → 후보 {len(cand)}개, 희석 공시 점검 중...")
    cand['dilution'] = [dilution_check(c) for c in cand['corp_code']]
    shadow = shadow.merge(cand[['ticker', 'dilution']], on='ticker', how='left')
    kr_results[grp] = {'all': sc, 'cand': kr_table(cand), 'shadow': kr_table(shadow)}

    base = f"{SAVE_DIR}/KR_{grp}_{YEAR}_{REPRT}_{SNAP_STR}"
    kr_results[grp]['cand'].to_csv(f"{base}_후보{KR_N_CANDIDATES}.csv", index=False, encoding='utf-8-sig')
    kr_results[grp]['shadow'].to_csv(f"{base}_그림자{KR_N_SHADOW}.csv", index=False, encoding='utf-8-sig')
    sc.to_csv(f"{base}_전체점수.csv", index=False, encoding='utf-8-sig')

    print(f"\n{'='*100}\n  {grp}주 후보 {KR_N_CANDIDATES}  ({YEAR} {REPORT_NAME} TTM / 가격 {SNAP_STR})\n{'='*100}")
    show = ['순위', '종목명', '시장', '업종', '시총(억)', 'POR', 'PBR', 'PSR', 'ROE(%)',
            '영업이익증가/자산(%)', '12-7개월(%)', '최근1개월(%)', '총점', '최근1년 희석공시', '점검']
    print(kr_results[grp]['cand'][[c for c in show if c in kr_results[grp]['cand'].columns]].to_string(index=False))


  대형: 점수화 312개 → 후보 30개, 희석 공시 점검 중...

  대형주 후보 30  (2026 반기보고서 TTM / 가격 20261102)
 순위        종목명     시장                                    업종      시총(억)   POR   PBR   PSR  ROE(%)  영업이익증가/자산(%)  12-7개월(%)  최근1개월(%)   총점 최근1년 희석공시 점검
  1    아이티센글로벌 KOSDAQ               컴퓨터 프로그래밍, 시스템 통합 및 관리업     4757.0  1.43  0.77  0.04    42.6           3.4      144.5     -29.7 0.90       CB1   
  2       삼지전자 KOSDAQ                        통신 및 방송 장비 제조업     4447.0  0.74  0.40  0.06    39.8          15.7      169.4     -12.1 0.90             
  3     에스에이엠티 KOSDAQ                      기계장비 및 관련 물품 도매업     7420.0  1.36  0.89  0.12    47.6          18.6      103.0     -11.1 0.86             
  4       자화전자  KOSPI                              전자부품 제조업     4032.0  4.32  0.76  0.40    21.2           5.8      107.9     -30.1 0.83             
  5 HD현대에너지솔루션  KOSPI                               반도체 제조업     8310.0  8.81  1.76  1.39    17.9           8.3      300.4     -38.7 0.82             
  6     SK하이닉스 

In [10]:
# ============================================================
# AI 정성검토용 마크다운 내보내기 (한국/미국 공용)
# ============================================================
def _md(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:          # tabulate 미설치 시
        return '```\n' + df.to_string(index=False) + '\n```'

KR_REVIEW_GUIDE = """\
## 검토 방식 (제외 사유 찾기)
후보는 퀀트 점수순입니다. 정성 검토는 '좋은 종목 고르기'가 아니라 **아래 사유에 해당하는 종목 빼기**로 하고,
남은 종목 중 총점 순으로 최종 종목을 정합니다. 제외 사유는 기록해 둡니다(나중에 그림자 포트폴리오와 비교).

1. 이익의 지속성: 최근 이익 급증이 원자재 가격·환율·일회성 수주/자산매각 때문인가?
2. 희석 위험: 최근 1년 CB/BW/유상증자, 미전환 CB 잔량, 리픽싱 조건
3. 지배구조: 최대주주 변경, 횡령·배임, 감사의견, 내부자 대량 매도
4. 업황 사이클: 업황 정점 신호(재고 증가, 가격 하락 전환)
5. 집중도: 같은 테마/원자재에 노출된 종목이 과도하게 겹치는가?
"""

US_REVIEW_GUIDE = """\
## 검토 방식 (제외 사유 찾기)
후보는 퀀트 점수순입니다. 아래 사유에 해당하는 종목을 빼고, 남은 종목 중 총점 순으로 최종 종목을 정합니다.

1. 마진이 '일시적으로' 눌린 것인가, 구조적으로 무너진 것인가? (피셔 PSR의 핵심 전제)
2. 대규모 소송·규제·특허 만료(제약), 회계 이슈
3. 부채 부담·신용등급 하락, 배당/자사주 축소
4. 진행 중인 M&A·분사·구조조정으로 재무 수치가 왜곡되었는가?
5. 섹터/테마 쏠림
"""

def export_review_md(title, basis, sections, guide, path):
    """sections: [(소제목, DataFrame), ...]"""
    lines = [f"# {title}", "", basis, "", guide, ""]
    for sub, df in sections:
        lines += [f"## {sub}", "", _md(df), ""]
    with open(path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"  AI 검토용 파일: {path}")


if 'kr_results' in globals() and kr_results:
    kr_cols = ['순위', '종목코드', '종목명', '시장', '업종', '주요제품', '시총(억)', 'PER', 'PBR', 'PSR', 'POR',
               'ROE(%)', '영업이익률(%)', '매출성장(%)', '영업이익증가/자산(%)', '12-7개월(%)', '최근1개월(%)',
               '거래대금20d(억)', '가치', '퀄리티', '성장', '모멘텀', '총점', '최근1년 희석공시', '점검']
    secs = [(f"{g}주 후보 {KR_N_CANDIDATES}", v['cand'][[c for c in kr_cols if c in v['cand'].columns]])
            for g, v in kr_results.items()]
    export_review_md(
        f"한국 후보 리스트 ({SNAP_STR})",
        f"재무: {YEAR}년 {REPORT_NAME} 기준 TTM / 가격: {SNAP_STR} / 규모: 시장 전체 시총 상위·하위 {KR_SIZE_Q:.0%}\n"
        f"점수 = 가치(1/POR,1/PBR,1/PSR) + 퀄리티(영업이익/자산, ROE) + 성장(영업이익증가/자산, 매출성장) "
        f"+ 모멘텀(12~7개월) + 반전(최근1개월, 0.5배). 각 0~1 백분위, 총점 높을수록 상위.",
        secs, KR_REVIEW_GUIDE, f"{SAVE_DIR}/AI검토_KR_{SNAP_STR}.md")

  AI 검토용 파일: /content/drive/MyDrive/Investment/Rebalancing_portfolio/AI검토_KR_20261102.md


## 미국

In [11]:
# ============================================================
# [US 1] S&P500 펀더멘털 수집 (yfinance)
# ============================================================
# 원본과의 차이:
#  - R&D 없는 종목을 버리지 않음 (yfinance에 R&D 줄이 없는 것은 데이터 누락인 경우도 많음)
#  - 금융/유틸리티/리츠만 제외 (PSR이 의미 없는 업종)
#  - 연간 손익 3~5년치로 '평균 영업마진' 계산 → 정규화 PSR(피셔의 핵심 논리)에 사용
us_cache = f'{SAVE_DIR}/us_fund_{SNAP_STR}.csv'
sp = fdr.StockListing('S&P500')
sp = sp[~sp['Sector'].isin(US_EXCLUDE_SECTORS)].copy()
sp['yf'] = sp['Symbol'].str.replace('.', '-', regex=False)   # BF.B → BF-B

if os.path.exists(us_cache):
    us_raw = pd.read_csv(us_cache)
    print(f"[US 1] 캐시 로드: {us_cache} ({len(us_raw)}개)")
else:
    print(f"[US 1] {len(sp)}개 종목 펀더멘털 수집...")
    recs = []
    for _, r in tqdm(sp.iterrows(), total=len(sp)):
        rec = {'Symbol': r['Symbol'], 'yf': r['yf'], 'Name': r['Name'], 'Sector': r['Sector']}
        try:
            t = yf.Ticker(r['yf'])
            info = t.info
            rec['marcap'] = info.get('marketCap')
            rec['PSR'] = info.get('priceToSalesTrailing12Months')
            rec['opm_ttm'] = info.get('operatingMargins')
            rec.update(us_fin_summary(t.financials, US_MIN_YEARS))
        except Exception as e:
            rec['err'] = type(e).__name__
        recs.append(rec)
        time.sleep(0.1)
    us_raw = pd.DataFrame(recs)
    us_raw.to_csv(us_cache, index=False)
    print(f"  저장: {us_cache}")

[US 1] 캐시 로드: /content/drive/MyDrive/Investment/Rebalancing_portfolio/us_fund_20261102.csv (366개)


In [12]:
# ============================================================
# [US 2] 피셔 PSR+PRR 보완 점수 → 후보 30 + 그림자 10
# ============================================================
#  value (피셔):
#    NEY       = 평균 영업마진 / PSR  : '정상 마진'으로 환산한 이익수익률. 마진이 일시적으로 눌려
#                PSR이 낮은 회사를 찾는다는 피셔의 원래 논리를 숫자로 옮긴 것
#    SP_sector = 섹터 안에서의 1/PSR 순위 : 저마진 업종(하드웨어·화학)이 PSR만으로 상위에 몰리는 것 방지
#    RDY       = R&D / 시총 (=1/PRR), IT·헬스케어만 : 피셔가 PRR을 쓰던 기술/제약 기업에 한정
#  momentum : 12-1개월 수익률 (미국은 근거가 탄탄한 표준 모멘텀)
#  value 1 : momentum 1 통합 점수 (가치·모멘텀은 상관이 음수라 같이 쓰면 분산 효과)
US_SHADOW_MAX_PER_SECTOR = 3
d = us_raw.copy()
for c in ['marcap', 'PSR', 'avg_op_margin', 'rd_last', 'n_years', 'opm_ttm']:
    d[c] = pd.to_numeric(d.get(c), errors='coerce')

print("[US 2] 필터...")
print(f"  {'S&P500 (금융/유틸/리츠 제외)':<34s}: {len(d):5d}")
d = log_step(d, (d['marcap'] > 0) & (d['PSR'] > 0), '시총·PSR 유효')
d = log_step(d, d['avg_op_margin'].notna(), f'연간 손익 {US_MIN_YEARS}년 이상')
d = log_step(d, d['avg_op_margin'] >= US_MIN_AVG_MARGIN, f'평균 영업마진 {US_MIN_AVG_MARGIN:.0%} 이상')

# 가격 → 12-1 모멘텀 (일괄 다운로드)
print("  가격 다운로드...")
px_all = yf.download(d['yf'].tolist(), period='15mo', auto_adjust=True,
                     progress=False, threads=True)['Close']
mom = {s: price_features(pd.DataFrame({'Close': px_all[s].dropna()}))['ret_12_1']
       if s in px_all.columns else np.nan for s in d['yf']}
d['ret_12_1'] = d['yf'].map(mom)

d['NEY'] = d['avg_op_margin'] / d['PSR']
d['SP_sector'] = d.groupby('Sector')['PSR'].rank(pct=True, ascending=False)
d['RDY'] = np.where(d['Sector'].isin(US_RD_SECTORS) & (d['rd_last'] > 0),
                    d['rd_last'] / d['marcap'], np.nan)
d['PRR'] = np.where(d['rd_last'] > 0, d['marcap'] / d['rd_last'], np.nan)

us_all = composite_scores(d, US_FACTORS, US_WEIGHTS)
us_cand = pick_with_cap(us_all, US_N_CANDIDATES, 'Sector', US_MAX_PER_SECTOR)
us_shadow = pick_with_cap(us_all, US_N_SHADOW, 'Sector', US_SHADOW_MAX_PER_SECTOR)

US_SHOW = {'rank_total': 'Rank', 'Symbol': 'Symbol', 'Name': 'Name', 'Sector': 'Sector',
           'mcap_B': 'MktCap($B)', 'PSR': 'PSR', 'avgOPM_%': 'AvgOPM(%)', 'opm_ttm_%': 'OPM_TTM(%)',
           'normPE': 'NormPE', 'PRR': 'PRR', 'mom_%': 'Mom12-1(%)', 'S_value': 'Value',
           'S_momentum': 'Momentum', 'S_total': 'Total', 'last_fy': 'LastFY'}

def us_table(x):
    t = x.copy()
    t['mcap_B'] = (t['marcap'] / 1e9).round(1)
    t['avgOPM_%'] = (t['avg_op_margin'] * 100).round(1)
    t['opm_ttm_%'] = (t['opm_ttm'] * 100).round(1)
    t['normPE'] = (1 / t['NEY']).round(1)          # 정상 마진 기준 PER(세전 영업이익 기준)
    t['mom_%'] = (t['ret_12_1'] * 100).round(1)
    for c in ['PSR', 'PRR', 'S_value', 'S_momentum', 'S_total']:
        t[c] = pd.to_numeric(t[c], errors='coerce').round(2)
    return t[[c for c in US_SHOW if c in t.columns]].rename(columns=US_SHOW).reset_index(drop=True)

us_cand_t, us_shadow_t = us_table(us_cand), us_table(us_shadow)
base = f"{SAVE_DIR}/US_{SNAP_STR}"
us_cand_t.to_csv(f"{base}_후보{US_N_CANDIDATES}.csv", index=False)
us_shadow_t.to_csv(f"{base}_그림자{US_N_SHADOW}.csv", index=False)
us_all.to_csv(f"{base}_전체점수.csv", index=False)

print(f"\n{'='*100}\n  US 후보 {US_N_CANDIDATES} ({SNAP_STR})\n{'='*100}")
print(us_cand_t.to_string(index=False))
stale = us_cand_t[pd.to_datetime(us_cand_t['LastFY'], errors='coerce') <
                  pd.Timestamp(RUN_DATE) - pd.Timedelta(days=550)]
if len(stale):
    print(f"\n⚠ 최신 연간 재무가 1.5년 이상 지난 종목 {len(stale)}개: {', '.join(stale['Symbol'])}")

export_review_md(
    f"US Candidates ({SNAP_STR})",
    "Fisher PSR+PRR 보완: Value = 정상마진 이익수익률(평균 영업마진/PSR) + 섹터 내 1/PSR + R&D/시총(IT·헬스케어) / "
    "Momentum = 12-1개월. Value:Momentum = 1:1. 각 0~1 백분위, Total 높을수록 상위. "
    "NormPE = 평균 영업마진 기준 PER(세전).",
    [(f"Candidates {US_N_CANDIDATES}", us_cand_t)], US_REVIEW_GUIDE, f"{SAVE_DIR}/AI검토_US_{SNAP_STR}.md")

[US 2] 필터...
  S&P500 (금융/유틸/리츠 제외)              :   366
  시총·PSR 유효                         :   366 →   365
  연간 손익 3년 이상                       :   365 →   363
  평균 영업마진 5% 이상                     :   363 →   314
  가격 다운로드...

  US 후보 30 (20261102)
 Rank Symbol                       Name                 Sector  MktCap($B)  PSR  AvgOPM(%)  OPM_TTM(%)  NormPE    PRR  Mom12-1(%)  Value  Momentum  Total     LastFY
    1    VLO              Valero Energy                 Energy       111.5 0.76        5.9        20.5    12.9    NaN       144.4   0.86      0.98   0.92 2025-12-31
    2    MPC         Marathon Petroleum                 Energy       129.8 0.84        6.8        13.6    12.3    NaN       118.2   0.84      0.96   0.90 2025-12-31
    3    DAL            Delta Air Lines            Industrials        52.7 0.73        8.9         7.2     8.2    NaN        44.3   0.94      0.85   0.90 2025-12-31
    4    DVA                     DaVita            Health Care        11.8 0.83       13.6 

In [13]:
# ============================================================
# [US 3] ETF 슬리브: QQQM vs VEA vs VWO (VAA 13612W 모멘텀), 전부 음수면 BIL
# ============================================================
def select_winner_etf(tickers=('QQQM', 'VEA', 'VWO'), defensive='BIL'):
    px = yf.download(list(tickers), period='14mo', auto_adjust=True, progress=False)['Close'].dropna()
    if len(px) < 21 * 12 + 1:
        raise ValueError(f"데이터 부족: {len(px)}일")
    last = px.iloc[-1]
    score = (12 * (last / px.iloc[-1 - 21] - 1) + 4 * (last / px.iloc[-1 - 63] - 1) +
             2 * (last / px.iloc[-1 - 126] - 1) + 1 * (last / px.iloc[-1 - 252] - 1)) * 100
    score = score.sort_values(ascending=False)
    print("=== ETF 모멘텀 (VAA 가중) ===")
    print(score.round(2).to_string())
    winner = defensive if score.max() < 0 else score.idxmax()
    print(f"→ ETF 슬리브: {winner}" + (" (전부 음수 → 방어)" if winner == defensive else ""))
    return winner, score

etf_winner, etf_scores = select_winner_etf()

=== ETF 모멘텀 (VAA 가중) ===
Ticker
VEA      -1.27
QQQM    -73.91
VWO    -104.29
→ ETF 슬리브: BIL (전부 음수 → 방어)


## 보유종목 점검 (버퍼 규칙)

In [14]:
# ============================================================
# [보유종목 점검] 버퍼 규칙: 보유 종목이 아직 후보권(순위 ≤ 30) 안이면 유지, 밖이면 교체
# ============================================================
# 매 리밸런싱마다 10개를 새로 뽑지 않고, 떨어진 종목만 교체 → 매매 횟수·관리 부담·거래비용 감소
# (상위 10 진입 기준과 30위 이탈 기준 사이의 '완충 구간'이 불필요한 회전을 막음)
HOLD_KR = []           # 현재 보유 한국 종목코드, 예: ['142210', '025820']
HOLD_US = []           # 현재 보유 미국 티커, 예: ['HPQ', 'GEHC']
KEEP_RANK = 30

def check_holdings_kr(hold):
    rows = []
    for t in hold:
        nm = mkt_df.loc[mkt_df['ticker'] == t, 'name']
        r = {'종목코드': t, '종목명': nm.iloc[0] if len(nm) else '(상장목록에 없음)'}
        grp = uni.loc[uni['ticker'] == t, 'size_group'].squeeze() if (uni['ticker'] == t).any() else None
        hit = [(g, v['all']) for g, v in kr_results.items() if t in set(v['all']['ticker'])]
        if hit:
            g, sc = hit[0]
            rk = int(sc.loc[sc['ticker'] == t, 'rank_total'].iloc[0])
            r.update({'구간': g, '순위': rk, '판단': '유지' if rk <= KEEP_RANK else f'교체 (순위 {rk} > {KEEP_RANK})'})
        elif t not in set(uni['ticker']):
            r.update({'판단': '교체 검토: 유니버스 제외(금융/관리종목 등)'})
        elif grp is None or (isinstance(grp, float) and np.isnan(grp)):
            r.update({'판단': '교체 검토: 시총 상·하위 20% 구간 이탈'})
        else:
            r.update({'구간': grp, '판단': '교체: 재무/유동성 필터 탈락 (적자전환·일회성이익·거래대금 등)'})
        rows.append(r)
    return pd.DataFrame(rows)

def check_holdings_us(hold):
    rows = []
    for s in hold:
        x = us_all[us_all['Symbol'] == s]
        if len(x):
            rk = int(x['rank_total'].iloc[0])
            rows.append({'Symbol': s, 'Rank': rk, '판단': '유지' if rk <= KEEP_RANK else f'교체 (순위 {rk} > {KEEP_RANK})'})
        else:
            rows.append({'Symbol': s, 'Rank': None, '판단': '교체: 필터 탈락 (평균마진 5% 미만/데이터 없음/업종 제외)'})
    return pd.DataFrame(rows)

if HOLD_KR:
    hk = check_holdings_kr(HOLD_KR)
    print(f"[한국 보유 점검] 유지 {int((hk['판단'] == '유지').sum())} / {len(hk)}")
    print(hk.to_string(index=False))
if HOLD_US:
    hu = check_holdings_us(HOLD_US)
    print(f"\n[미국 보유 점검] 유지 {int((hu['판단'] == '유지').sum())} / {len(hu)}")
    print(hu.to_string(index=False))
if HOLD_KR or HOLD_US:
    print("\n→ '유지'는 그대로 두고, 빈 자리만 후보 리스트 상위(정성 검토 통과)에서 채우면 됩니다.")

## 최종 확정 (AI 검토 후 입력)

In [15]:
# ============================================================
# [최종] AI 검토 후 확정한 종목 입력 → 매수 금액 / 유동성 점검 / 기록
# ============================================================
# 검토 후 아래 리스트를 채우고 실행. 제외한 종목과 사유도 남겨두면 나중에 그림자 포트폴리오와 비교 가능.
FINAL_KR = []          # 예: ['142210', '025820', ...]  (대형+소형 섞어서 10개)
FINAL_US = []          # 예: ['HPQ', 'GEHC', ...]
EXCLUDED = {}          # 예: {'012800': '동 가격 상승에 따른 일시적 이익', 'BAX': '구조조정 진행중'}
TOTAL_KR = None  # 비공개: 실제 값은 원본 노트북(private)에만 둠
TOTAL_US = None  # 비공개: 실제 값은 원본 노트북(private)에만 둠

if FINAL_KR:
    allk = pd.concat([v['all'] for v in kr_results.values()])
    sel = allk[allk['ticker'].isin(FINAL_KR)][['ticker', 'name', 'size_group', 'S_total', 'tv20', 'close']].copy()
    per = TOTAL_KR / len(FINAL_KR)
    sel['매수금액'] = per
    sel['주수(대략)'] = (per // sel['close']).astype(int)
    sel['투자금/일거래대금(%)'] = (per / sel['tv20'] * 100).round(2)
    print(sel.to_string(index=False))
    sel.to_csv(f"{SAVE_DIR}/최종매수_KR_{SNAP_STR}.csv", index=False, encoding='utf-8-sig')

if FINAL_US and 'us_all' in globals():
    sel = us_all[us_all['Symbol'].isin(FINAL_US)][['Symbol', 'Name', 'Sector', 'S_total']].copy()
    sel['매수금액(원)'] = TOTAL_US / len(FINAL_US)
    print(sel.to_string(index=False))
    sel.to_csv(f"{SAVE_DIR}/최종매수_US_{SNAP_STR}.csv", index=False, encoding='utf-8-sig')

if EXCLUDED:
    pd.Series(EXCLUDED, name='제외사유').rename_axis('종목').to_csv(
        f"{SAVE_DIR}/제외사유_{SNAP_STR}.csv", encoding='utf-8-sig')
    print(f"제외 사유 {len(EXCLUDED)}건 저장")